# 06 · Example predictions

Train the selected XGBoost pipeline on batch 0 using the same grouped split as notebook 05. It excludes `status` and `prev_sold_date`. This is an exploratory model, not an MLflow champion.


## Imports

In [ ]:
import json
import sys
import os
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path('../src').resolve()))
from property_ml.features import RAW_FEATURES, build_model, prepare
from property_ml.splitting import assign_splits


## Load batch

In [ ]:
DATA = Path('../data')
GROUP = int(os.environ['DATA_API_GROUP'])
files = sorted(DATA.glob(f'group-{GROUP}-*.json'))
if not files:
    raise FileNotFoundError('Collect a batch in notebook 01 first.')
with files[0].open() as file:
    batch = json.load(file)
df = pd.DataFrame(batch['data'])
print(f"Group {GROUP} | batch {batch['batch_number']} | {len(df):,} rows")


## Prepare and split

In [ ]:
data = df.loc[df['price'].notna() & df['price'].gt(0)].copy()
raw_features = RAW_FEATURES
X, y = data[RAW_FEATURES].copy(), data['price']
splits = assign_splits(data)
X_train, X_test = X.loc[splits == 'train'], X.loc[splits == 'test']
y_train, y_test = y.loc[splits == 'train'], y.loc[splits == 'test']
print('Training / held-out test:', len(X_train), len(X_test))


## Train model

In [ ]:
model = build_model()
model.fit(X_train, y_train)
print('Selected XGBoost pipeline ready.')


## Predict held-out properties

In [ ]:
examples = X_test.sample(n=min(5, len(X_test)), random_state=42)
comparison = examples[raw_features].copy()
comparison['actual_price'] = y_test.loc[examples.index]
comparison['predicted_price'] = model.predict(examples)
comparison['absolute_error'] = (comparison['actual_price'] - comparison['predicted_price']).abs()
display(comparison.round(2))


## Try one property

Edit the input values; `price` is not an input.

In [ ]:
example = data.loc[X_test.index[:1], raw_features].copy()
# Example: example.loc[:, 'bed'] = 3
print('Inputs:')
display(example)
predicted_price = model.predict(example)[0]
print(f'Predicted dataset price: {predicted_price:,.2f}')


The target can be a listing or recent sale price. These sample predictions are for exploration; they are not a trained MLflow champion or a guarantee of a sale price.